# XFeat BRIEF-channel filter + Quantile-ABS reranking

This separate experiment applies the BRIEF principle to **pairs of XFeat descriptor channels**. It selects 256 stable, balanced channel comparisons offline; packs them into 32 bytes; performs all-pairs Hamming candidate search; and reranks only top-K candidates with the 8-level quantile-ABS distance.

The notebook reports: full float cosine, exhaustive quantile-ABS, BRIEF-only Hamming, hybrid top-K accuracy, Hamming Recall@K, matching-only latency, online encoding+matching latency, and storage. The uploaded images are used to generate separate training and held-out synthetic homographies with known ground truth. This is a prototype evaluation, not a claim of novelty or a full dataset benchmark.


In [ ]:
!pip -q install opencv-python-headless numba

import time
import cv2
import torch
import numpy as np
import torch.nn.functional as F

from numba import njit, prange
from google.colab import files

SEED = 7
TOP_K = 1000
LEVELS = 8
BRIEF_BITS = 256
HYBRID_K_VALUES = (5, 10, 20)
TRAIN_WARPS_PER_IMAGE = 5
VALIDATION_WARPS_PER_IMAGE = 2
MAX_POSITIVES_PER_PAIR = 600
GEOMETRIC_THRESHOLD = 3.0

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

xfeat = torch.hub.load(
    "verlab/accelerated_features",
    "XFeat",
    pretrained=True,
    top_k=TOP_K,
).to(device).eval()

for parameter in xfeat.parameters():
    parameter.requires_grad_(False)

print("Device:", device)
print("Upload one or more textured JPG/PNG images.")
print("Synthetic homographies create known ground truth; two real views alone do not provide exact ground truth.")
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload at least one JPG or PNG image.")


def decode_and_resize(raw_bytes, maximum_side=640):
    image = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError("One uploaded file is not a readable JPG/PNG image.")
    height, width = image.shape[:2]
    scale = min(1.0, maximum_side / max(height, width))
    new_width = max(64, (int(width * scale) // 32) * 32)
    new_height = max(64, (int(height * scale) // 32) * 32)
    return cv2.resize(image, (new_width, new_height))


images = [decode_and_resize(raw) for raw in uploaded.values()]
print("Loaded images:", len(images))
print("Image sizes:", [image.shape[:2] for image in images])


In [ ]:
def image_to_tensor(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float()[None] / 255.0
    return tensor.to(device)


@torch.inference_mode()
def extract_features(image):
    output = xfeat.detectAndCompute(image_to_tensor(image), top_k=TOP_K)[0]
    return {
        "keypoints": output["keypoints"].detach().cpu().float(),
        "descriptors": F.normalize(output["descriptors"].detach().cpu().float(), dim=-1),
    }


def make_random_homography(height, width, generator, difficulty=1.0):
    center = (width / 2.0, height / 2.0)
    angle = generator.uniform(-20.0, 20.0) * difficulty
    scale = generator.uniform(0.88, 1.08)
    affine = cv2.getRotationMatrix2D(center, angle, scale)
    affine[0, 2] += generator.uniform(-0.06, 0.06) * width * difficulty
    affine[1, 2] += generator.uniform(-0.06, 0.06) * height * difficulty
    homography = np.vstack([affine, [0.0, 0.0, 1.0]]).astype(np.float32)
    homography[2, 0] = generator.uniform(-7e-5, 7e-5) * difficulty
    homography[2, 1] = generator.uniform(-7e-5, 7e-5) * difficulty
    return homography


def warp_image(image, homography, generator):
    height, width = image.shape[:2]
    warped = cv2.warpPerspective(
        image, homography, (width, height), flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT,
    )
    alpha = float(generator.uniform(0.82, 1.12))
    beta = float(generator.uniform(-12.0, 18.0))
    return cv2.convertScaleAbs(warped, alpha=alpha, beta=beta)


def project_points(points, homography):
    points_np = points.numpy()
    homogeneous = np.concatenate(
        [points_np, np.ones((len(points_np), 1), dtype=np.float32)], axis=1
    )
    projected = (homography @ homogeneous.T).T
    projected = projected[:, :2] / np.clip(projected[:, 2:3], 1e-8, None)
    return torch.from_numpy(projected.astype(np.float32))


def ground_truth_pairs(keypoints_1, keypoints_2, homography, threshold=GEOMETRIC_THRESHOLD):
    projected_12 = project_points(keypoints_1, homography)
    distances_12 = torch.cdist(projected_12, keypoints_2)
    nearest_12 = distances_12.argmin(dim=1)
    errors_12 = distances_12[torch.arange(len(keypoints_1)), nearest_12]

    inverse_h = np.linalg.inv(homography).astype(np.float32)
    projected_21 = project_points(keypoints_2, inverse_h)
    distances_21 = torch.cdist(projected_21, keypoints_1)
    nearest_21 = distances_21.argmin(dim=1)

    indices_1 = torch.arange(len(keypoints_1))
    good = (nearest_21[nearest_12] == indices_1) & (errors_12 < threshold)
    return indices_1[good], nearest_12[good]


def build_pairs(source_images, train_warps, validation_warps, generator):
    training_pairs, validation_pairs = [], []
    for image_index, image in enumerate(source_images):
        base = extract_features(image)
        for warp_index in range(train_warps + validation_warps):
            homography = make_random_homography(*image.shape[:2], generator=generator)
            warped_image = warp_image(image, homography, generator)
            warped = extract_features(warped_image)
            index_1, index_2 = ground_truth_pairs(
                base["keypoints"], warped["keypoints"], homography
            )
            if len(index_1) > MAX_POSITIVES_PER_PAIR:
                keep = torch.randperm(len(index_1))[:MAX_POSITIVES_PER_PAIR]
                index_1, index_2 = index_1[keep], index_2[keep]
            record = {
                "descriptors_1": base["descriptors"],
                "descriptors_2": warped["descriptors"],
                "keypoints_1": base["keypoints"],
                "keypoints_2": warped["keypoints"],
                "positive_indices_1": index_1,
                "positive_indices_2": index_2,
                "homography": homography,
            }
            target = training_pairs if warp_index < train_warps else validation_pairs
            target.append(record)
            print(
                f"image={image_index} warp={warp_index} split={'train' if target is training_pairs else 'valid'} "
                f"positives={len(index_1)}"
            )
    return training_pairs, validation_pairs


start = time.perf_counter()
training_pairs, validation_pairs = build_pairs(
    images, TRAIN_WARPS_PER_IMAGE, VALIDATION_WARPS_PER_IMAGE, rng
)
dataset_time = time.perf_counter() - start

training_anchors = torch.cat([
    pair["descriptors_1"][pair["positive_indices_1"]] for pair in training_pairs
])
training_positives = torch.cat([
    pair["descriptors_2"][pair["positive_indices_2"]] for pair in training_pairs
])

if len(training_anchors) < 128:
    raise RuntimeError("Too few geometric positives. Upload a more textured image or more images.")

print("Training pairs:", len(training_pairs))
print("Validation pairs:", len(validation_pairs))
print("Calibration positives:", len(training_anchors))
print(f"Feature dataset construction: {dataset_time:.2f} s")


In [ ]:
def fit_quantile_edges(descriptors, levels=LEVELS):
    quantiles = torch.arange(1, levels, dtype=descriptors.dtype) / levels
    return torch.quantile(descriptors, quantiles, dim=0).T.contiguous()


calibration_descriptors = torch.cat([training_anchors, training_positives], dim=0)
start = time.perf_counter_ns()
quantile_edges = fit_quantile_edges(calibration_descriptors)
quantile_calibration_ms = (time.perf_counter_ns() - start) / 1e6


def hard_quantize(descriptors):
    levels = (descriptors[:, :, None] > quantile_edges[None, :, :]).sum(dim=-1)
    return np.ascontiguousarray(levels.numpy(), dtype=np.uint8)


def all_channel_pairs(dimensions=64):
    left, right = np.triu_indices(dimensions, k=1)
    return np.stack([left, right], axis=1).astype(np.int16)


def select_brief_channel_tests(anchors, positives, number_of_bits=BRIEF_BITS):
    # Supervised BRIEF-style test selection. Positive pairs should preserve a bit;
    # random negatives should disagree. Balanced, weakly correlated bits are preferred.
    sample_count = min(6000, len(anchors))
    sample_indices = rng.choice(len(anchors), sample_count, replace=False)
    anchor = anchors[sample_indices].numpy()
    positive = positives[sample_indices].numpy()
    negative = positive[rng.permutation(sample_count)]

    candidates = all_channel_pairs(anchor.shape[1])
    left, right = candidates[:, 0], candidates[:, 1]
    anchor_bits = anchor[:, left] > anchor[:, right]
    positive_bits = positive[:, left] > positive[:, right]
    negative_bits = negative[:, left] > negative[:, right]

    positive_stability = (anchor_bits == positive_bits).mean(axis=0)
    negative_discrimination = (anchor_bits != negative_bits).mean(axis=0)
    bit_probability = np.concatenate([anchor_bits, positive_bits], axis=0).mean(axis=0)
    balance = 1.0 - 2.0 * np.abs(bit_probability - 0.5)
    score = 0.55 * positive_stability + 0.30 * negative_discrimination + 0.15 * balance

    # Correlation screening is limited to the strongest candidates for predictable runtime.
    ranked = np.argsort(-score)
    shortlist = ranked[:min(768, len(ranked))]
    correlation_rows = min(512, sample_count)
    patterns = np.concatenate(
        [anchor_bits[:correlation_rows], positive_bits[:correlation_rows]], axis=0
    )[:, shortlist].astype(np.float32)
    patterns -= patterns.mean(axis=0, keepdims=True)
    patterns /= np.maximum(patterns.std(axis=0, keepdims=True), 1e-6)

    selected_local = []
    selected_patterns = None
    for local_index in range(len(shortlist)):
        pattern = patterns[:, local_index]
        if selected_patterns is not None:
            correlations = np.abs(pattern @ selected_patterns) / len(pattern)
            if correlations.max() >= 0.90:
                continue
        selected_local.append(local_index)
        selected_patterns = patterns[:, selected_local]
        if len(selected_local) == number_of_bits:
            break

    selected_global = list(shortlist[selected_local])
    if len(selected_global) < number_of_bits:
        selected_set = set(selected_global)
        selected_global.extend(
            index for index in ranked if index not in selected_set
        )
        selected_global = selected_global[:number_of_bits]
    selected_global = np.asarray(selected_global, dtype=np.int64)

    stats = {
        "stability": float(positive_stability[selected_global].mean()),
        "negative_disagreement": float(negative_discrimination[selected_global].mean()),
        "balance": float(balance[selected_global].mean()),
    }
    return np.ascontiguousarray(candidates[selected_global], dtype=np.int16), stats


start = time.perf_counter_ns()
brief_pairs, brief_stats = select_brief_channel_tests(training_anchors, training_positives)
brief_selection_ms = (time.perf_counter_ns() - start) / 1e6


def encode_brief(descriptors):
    values = descriptors.numpy()
    bits = values[:, brief_pairs[:, 0]] > values[:, brief_pairs[:, 1]]
    return np.ascontiguousarray(np.packbits(bits, axis=1, bitorder="little"), dtype=np.uint8)


print(f"Quantile calibration (offline): {quantile_calibration_ms:.3f} ms")
print(f"BRIEF test selection (offline): {brief_selection_ms:.3f} ms")
print("Selected channel tests:", len(brief_pairs))
print("Mean positive stability:", f"{brief_stats['stability']:.3f}")
print("Mean random-negative disagreement:", f"{brief_stats['negative_disagreement']:.3f}")
print("Mean bit balance score:", f"{brief_stats['balance']:.3f}")


In [ ]:
POPCOUNT_LUT = np.asarray([int(i).bit_count() for i in range(256)], dtype=np.uint8)


@njit(parallel=True, cache=False)
def pairwise_uint8_abs(levels_1, levels_2):
    number_1, number_2 = levels_1.shape[0], levels_2.shape[0]
    dimensions = levels_1.shape[1]
    distances = np.empty((number_1, number_2), dtype=np.uint16)
    for i in prange(number_1):
        for j in range(number_2):
            total = 0
            for dimension in range(dimensions):
                left = int(levels_1[i, dimension])
                right = int(levels_2[j, dimension])
                total += left - right if left >= right else right - left
            distances[i, j] = total
    return distances


@njit(parallel=True, cache=False)
def pairwise_hamming(packed_1, packed_2):
    number_1, number_2 = packed_1.shape[0], packed_2.shape[0]
    byte_count = packed_1.shape[1]
    distances = np.empty((number_1, number_2), dtype=np.uint16)
    for i in prange(number_1):
        for j in range(number_2):
            total = 0
            for byte_index in range(byte_count):
                total += int(POPCOUNT_LUT[packed_1[i, byte_index] ^ packed_2[j, byte_index]])
            distances[i, j] = total
    return distances


@njit(parallel=True, cache=False)
def abs_nearest_from_candidates(levels_1, levels_2, candidates):
    output = np.empty(levels_1.shape[0], dtype=np.int64)
    dimensions = levels_1.shape[1]
    for i in prange(levels_1.shape[0]):
        best_index = int(candidates[i, 0])
        best_distance = 1 << 30
        for candidate_position in range(candidates.shape[1]):
            j = int(candidates[i, candidate_position])
            total = 0
            for dimension in range(dimensions):
                left = int(levels_1[i, dimension])
                right = int(levels_2[j, dimension])
                total += left - right if left >= right else right - left
            if total < best_distance or (total == best_distance and j < best_index):
                best_distance = total
                best_index = j
        output[i] = best_index
    return output


def mutual_from_distance_matrix(distances):
    nearest_12 = distances.argmin(axis=1)
    nearest_21 = distances.argmin(axis=0)
    indices_1 = np.arange(distances.shape[0])
    mutual = nearest_21[nearest_12] == indices_1
    return indices_1[mutual], nearest_12[mutual]


def mutual_float_cosine(descriptors_1, descriptors_2):
    similarities = descriptors_1 @ descriptors_2.T
    nearest_12 = similarities.argmax(dim=1)
    nearest_21 = similarities.argmax(dim=0)
    indices_1 = torch.arange(len(descriptors_1))
    mutual = nearest_21[nearest_12] == indices_1
    return indices_1[mutual].numpy(), nearest_12[mutual].numpy()


def shortlist_candidates(distances, k):
    k = min(k, distances.shape[1])
    return np.ascontiguousarray(
        np.argpartition(distances, kth=k - 1, axis=1)[:, :k], dtype=np.int64
    )


def hybrid_mutual(levels_1, levels_2, hamming_distances, k):
    candidates_12 = shortlist_candidates(hamming_distances, k)
    candidates_21 = shortlist_candidates(hamming_distances.T, k)
    nearest_12 = abs_nearest_from_candidates(levels_1, levels_2, candidates_12)
    nearest_21 = abs_nearest_from_candidates(levels_2, levels_1, candidates_21)
    indices_1 = np.arange(len(levels_1))
    mutual = nearest_21[nearest_12] == indices_1
    return indices_1[mutual], nearest_12[mutual]


def encoded_pair(pair):
    d1, d2 = pair["descriptors_1"], pair["descriptors_2"]
    return hard_quantize(d1), hard_quantize(d2), encode_brief(d1), encode_brief(d2)


def geometric_errors(pair, indices_1, indices_2):
    projected = project_points(pair["keypoints_1"][indices_1], pair["homography"]).numpy()
    observed = pair["keypoints_2"][indices_2].numpy()
    return np.linalg.norm(projected - observed, axis=1)


def evaluate_method(name, matcher):
    all_errors = []
    for pair in validation_pairs:
        indices_1, indices_2 = matcher(pair)
        all_errors.append(geometric_errors(pair, indices_1, indices_2))
    errors = np.concatenate(all_errors) if all_errors else np.empty(0)
    correct = errors < GEOMETRIC_THRESHOLD
    result = {
        "name": name,
        "matches": len(errors),
        "correct": int(correct.sum()),
        "precision": 100.0 * correct.mean() if len(correct) else 0.0,
        "median_error": float(np.median(errors)) if len(errors) else float("nan"),
    }
    print(f"\n{name}")
    print("Mutual matches:", result["matches"])
    print("Correct matches:", result["correct"])
    print(f"Precision: {result['precision']:.2f}%")
    print(f"Median geometric error: {result['median_error']:.2f} pixels")
    return result


def match_float(pair):
    return mutual_float_cosine(pair["descriptors_1"], pair["descriptors_2"])


def match_full_abs(pair):
    q1, q2 = hard_quantize(pair["descriptors_1"]), hard_quantize(pair["descriptors_2"])
    return mutual_from_distance_matrix(pairwise_uint8_abs(q1, q2))


def match_hamming(pair):
    b1, b2 = encode_brief(pair["descriptors_1"]), encode_brief(pair["descriptors_2"])
    return mutual_from_distance_matrix(pairwise_hamming(b1, b2))


def match_hybrid(pair, k):
    q1, q2, b1, b2 = encoded_pair(pair)
    return hybrid_mutual(q1, q2, pairwise_hamming(b1, b2), k)


# Compile kernels outside all timings.
_ = pairwise_uint8_abs(np.zeros((2, 64), np.uint8), np.zeros((2, 64), np.uint8))
_ = pairwise_hamming(np.zeros((2, BRIEF_BITS // 8), np.uint8), np.zeros((2, BRIEF_BITS // 8), np.uint8))
_ = abs_nearest_from_candidates(
    np.zeros((2, 64), np.uint8), np.zeros((2, 64), np.uint8), np.zeros((2, 1), np.int64)
)

print("\n========== Held-out synthetic validation ==========")
results = [
    evaluate_method("A. Original XFeat float cosine", match_float),
    evaluate_method("B. Full quantile uint8 ABS", match_full_abs),
    evaluate_method("C. Selected BRIEF-channel Hamming", match_hamming),
]
for k in HYBRID_K_VALUES:
    results.append(
        evaluate_method(
            f"D. BRIEF Hamming top-{k} + quantile ABS rerank",
            lambda pair, k=k: match_hybrid(pair, k),
        )
    )


In [ ]:
def candidate_recall_report(k_values=(1, 5, 10, 20, 50)):
    totals = {k: {"gt_hit": 0, "gt_total": 0, "abs_hit": 0, "abs_total": 0} for k in k_values}
    for pair in validation_pairs:
        q1, q2, b1, b2 = encoded_pair(pair)
        hamming = pairwise_hamming(b1, b2)
        full_abs = pairwise_uint8_abs(q1, q2)
        abs_nearest = full_abs.argmin(axis=1)
        gt_1 = pair["positive_indices_1"].numpy()
        gt_2 = pair["positive_indices_2"].numpy()
        for k in k_values:
            candidates = shortlist_candidates(hamming, k)
            totals[k]["gt_hit"] += int((candidates[gt_1] == gt_2[:, None]).any(axis=1).sum())
            totals[k]["gt_total"] += len(gt_1)
            totals[k]["abs_hit"] += int((candidates == abs_nearest[:, None]).any(axis=1).sum())
            totals[k]["abs_total"] += len(abs_nearest)

    print("\n========== Hamming candidate quality ==========")
    print("GT Recall@K: known geometric positive appears in the Hamming shortlist.")
    print("ABS retention@K: exhaustive-ABS nearest neighbor appears in the shortlist.")
    for k in k_values:
        gt = 100.0 * totals[k]["gt_hit"] / max(totals[k]["gt_total"], 1)
        retained = 100.0 * totals[k]["abs_hit"] / max(totals[k]["abs_total"], 1)
        print(f"K={k:2d} | GT Recall@K={gt:6.2f}% | ABS retention@K={retained:6.2f}%")


candidate_recall_report()


def benchmark(function, warmup=5, repeats=20):
    for _ in range(warmup):
        function()
    values = []
    for _ in range(repeats):
        start = time.perf_counter_ns()
        function()
        values.append((time.perf_counter_ns() - start) / 1e6)
    values = np.asarray(values)
    return float(values.mean()), float(values.std()), float(np.median(values))


timing_pair = validation_pairs[0]
d1, d2 = timing_pair["descriptors_1"], timing_pair["descriptors_2"]
q1, q2, b1, b2 = encoded_pair(timing_pair)


def abs_match_only():
    return mutual_from_distance_matrix(pairwise_uint8_abs(q1, q2))


def hamming_match_only():
    return mutual_from_distance_matrix(pairwise_hamming(b1, b2))


def hybrid_match_only(k=20):
    return hybrid_mutual(q1, q2, pairwise_hamming(b1, b2), k)


def abs_online():
    left, right = hard_quantize(d1), hard_quantize(d2)
    return mutual_from_distance_matrix(pairwise_uint8_abs(left, right))


def hamming_online():
    left, right = encode_brief(d1), encode_brief(d2)
    return mutual_from_distance_matrix(pairwise_hamming(left, right))


def hybrid_online(k=20):
    left_q, right_q = hard_quantize(d1), hard_quantize(d2)
    left_b, right_b = encode_brief(d1), encode_brief(d2)
    return hybrid_mutual(left_q, right_q, pairwise_hamming(left_b, right_b), k)


runtime = {
    "float cosine match": benchmark(lambda: mutual_float_cosine(d1, d2)),
    "full ABS match": benchmark(abs_match_only),
    "full ABS online": benchmark(abs_online),
    "BRIEF Hamming match": benchmark(hamming_match_only),
    "BRIEF Hamming online": benchmark(hamming_online),
    "hybrid top-20 match": benchmark(lambda: hybrid_match_only(20)),
    "hybrid top-20 online": benchmark(lambda: hybrid_online(20)),
}

print("\n========== Runtime on one held-out pair ==========")
print("Feature extraction is excluded equally. JIT compilation is excluded.")
print("Offline calibration/test selection are reported separately and excluded from online latency.")
print("Descriptors:", tuple(d1.shape), "vs", tuple(d2.shape))
print("PyTorch CPU threads:", torch.get_num_threads())
for label, (mean, std, median) in runtime.items():
    print(f"{label:>25}: {mean:8.3f} +/- {std:6.3f} ms (median {median:8.3f} ms)")

float_median = runtime["float cosine match"][2]
print("\nMedian matching-only speedups vs float cosine")
for label in ("full ABS match", "BRIEF Hamming match", "hybrid top-20 match"):
    print(f"{label:>25}: {float_median / runtime[label][2]:.2f}x")

print("\nMedian online speedups vs float cosine matching")
for label in ("full ABS online", "BRIEF Hamming online", "hybrid top-20 online"):
    print(f"{label:>25}: {float_median / runtime[label][2]:.2f}x")

print("\n========== Storage per descriptor ==========")
print("Float32 XFeat: 256 bytes")
print("Quantile uint8 ABS code: 64 bytes")
print(f"Selected BRIEF code: {BRIEF_BITS // 8} bytes")
print(f"Hybrid stored naively (ABS + BRIEF): {64 + BRIEF_BITS // 8} bytes")
print("Note: packing each 8-level ABS coordinate into 3 bits could reduce the ABS code to 24 bytes,")
print("but this notebook intentionally times the simple uint8 ABS kernel.")


In [ ]:
artifact = {
    "brief_channel_pairs": torch.from_numpy(brief_pairs.astype(np.int64)),
    "quantile_edges": quantile_edges,
    "brief_bits": BRIEF_BITS,
    "quantile_levels": LEVELS,
    "seed": SEED,
}
torch.save(artifact, "xfeat_brief_abs_hybrid.pt")
print("Saved xfeat_brief_abs_hybrid.pt")
files.download("xfeat_brief_abs_hybrid.pt")
